# Data Preprocessing and Feature Engineering

## Purpose

This notebook prepares the cleaned Phase I diabetes dataset for the machine-learning experiments in Phase II. It begins by validating the input data and documenting the predictor types before any transformations are applied.

The later preprocessing steps will address the representation of binary, ordinal, continuous, and count-based variables, as well as the imbalance in the `Diabetes_binary` target. Any transformation or class-imbalance technique that learns information from the data will be applied only to the training set to prevent data leakage.


## 1. Load the Cleaned Dataset

The cleaned dataset produced during Phase I is loaded as the input for Phase II preprocessing. Initial checks are performed before any new transformations are applied.

In [1]:
from pathlib import Path
import pandas as pd

data_path = Path("../data/processed/diabetes_cleaned.csv")

df = pd.read_csv(data_path)

print(f"Dataset loaded from: {data_path}")
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

df.head()

Dataset loaded from: ../data/processed/diabetes_cleaned.csv
Rows: 253,680
Columns: 22


,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,...,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income,Diabetes_binary
0,1,1,1,40,1,0,0,0,0,1,...,0,5,18,15,1,0,9,4,3,0
1,0,0,0,25,1,0,0,1,0,0,...,1,3,0,0,0,0,7,6,1,0
2,1,1,1,28,0,0,0,0,1,0,...,1,5,30,30,1,0,9,4,8,0
3,1,0,1,27,0,0,0,1,1,1,...,0,2,0,0,0,0,11,3,6,0
4,1,1,1,24,0,0,0,1,1,1,...,0,2,3,0,0,0,11,5,4,0


## 2. Initial Data Validation

The structure and quality of the cleaned dataset are checked again before preprocessing. This confirms that the Phase I output has been loaded correctly and that no unexpected missing values or structural changes have been introduced.

In [2]:
validation_summary = pd.DataFrame({
    "Data_Type": df.dtypes.astype(str),
    "Missing_Values": df.isna().sum(),
    "Unique_Values": df.nunique()})

print(f"Number of rows: {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
print(f"Total missing values: {df.isna().sum().sum():,}")
print(f"Exact matching rows: {df.duplicated().sum():,}")

display(validation_summary)

Number of rows: 253,680
Number of columns: 22
Total missing values: 0
Exact matching rows: 24,206


,Data_Type,Missing_Values,Unique_Values
HighBP,int64,0,2
HighChol,int64,0,2
CholCheck,int64,0,2
BMI,int64,0,84
Smoker,int64,0,2
Stroke,int64,0,2
HeartDiseaseorAttack,int64,0,2
PhysActivity,int64,0,2
Fruits,int64,0,2
Veggies,int64,0,2


## 3. Target Distribution

The distribution of `Diabetes_binary` is checked because a large difference between the two classes can affect model training and make ordinary accuracy misleading.

In [3]:
target_summary = (
    df["Diabetes_binary"]
    .value_counts()
    .sort_index()
    .rename_axis("Diabetes_binary")
    .reset_index(name="Respondents"))

target_summary["Class"] = target_summary["Diabetes_binary"].map({
    0: "No diabetes",
    1: "Prediabetes/Diabetes"})

target_summary["Percentage"] = (target_summary["Respondents"] / len(df) * 100).round(2)

target_summary = target_summary[["Diabetes_binary", "Class", "Respondents", "Percentage"]]

display(target_summary.style.hide(axis="index"))

Diabetes_binary,Class,Respondents,Percentage
0,No diabetes,218334,86.070000
1,Prediabetes/Diabetes,35346,13.930000


### Interpretation

The cleaned dataset retains the class imbalance identified during Phase I. Most respondents belong to the no-diabetes class (`86.07%`), while the combined prediabetes/diabetes class represents only `13.93%` of the dataset.

This imbalance will be considered during model training and evaluation. Any resampling or class-balancing method will be applied only to the training data so that the test data continues to represent the original population distribution.

## 4. Predictor Types

The predictors are grouped according to how their values should be interpreted. Although every column is stored numerically in the CSV file, the numbers do not all represent the same type of information. Identifying these groups helps prevent inappropriate transformations during preprocessing.

In [4]:
binary_features = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost",
    "DiffWalk", "Sex"]

ordinal_features = ["GenHlth", "Age", "Education", "Income"]

continuous_features = ["BMI"]

count_features = ["MentHlth", "PhysHlth"]

target = "Diabetes_binary"

feature_groups = pd.DataFrame({
    "Feature_Group": [
        "Binary", "Ordinal", "Continuous", "Count-based", "Target"
    ],
    "Features": [
        ", ".join(binary_features),
        ", ".join(ordinal_features),
        ", ".join(continuous_features),
        ", ".join(count_features),
        target
    ],
    "Number_of_Features": [
        len(binary_features),
        len(ordinal_features),
        len(continuous_features),
        len(count_features),
        1
    ]
})

classified_columns = (
    binary_features
    + ordinal_features
    + continuous_features
    + count_features
    + [target])

print(f"Columns classified: {len(classified_columns)}")
print(f"Dataset columns: {df.shape[1]}")
print(f"All columns accounted for: {set(classified_columns) == set(df.columns)}")

display(feature_groups.style.hide(axis="index"))

Columns classified: 22
Dataset columns: 22
All columns accounted for: True


Feature_Group,Features,Number_of_Features
Binary,"HighBP, HighChol, CholCheck, Smoker, Stroke, HeartDiseaseorAttack, PhysActivity, Fruits, Veggies, HvyAlcoholConsump, AnyHealthcare, NoDocbcCost, DiffWalk, Sex",14
Ordinal,"GenHlth, Age, Education, Income",4
Continuous,BMI,1
Count-based,"MentHlth, PhysHlth",2
Target,Diabetes_binary,1


## 5. Validity of Category Codes and Value Ranges

Each binary and ordinal variable is checked against its documented set of permitted values. The physical health and mental health variables are also checked because they represent number of days and should range from 0 to 30.

In [5]:
expected_values = {
    **{feature: {0, 1} for feature in binary_features},
    "GenHlth": {1, 2, 3, 4, 5},
    "Age": set(range(1, 14)),
    "Education": set(range(1, 7)),
    "Income": set(range(1, 9)),
    "MentHlth": set(range(0, 31)),
    "PhysHlth": set(range(0, 31)),
    "Diabetes_binary": {0, 1}}

validity_results = []

for column, permitted in expected_values.items():
    observed = set(df[column].dropna().unique())
    invalid = observed - permitted

    validity_results.append({
        "Variable": column,
        "Observed_Min": df[column].min(),
        "Observed_Max": df[column].max(),
        "Invalid_Values": sorted(invalid) if invalid else "None",
        "Valid": len(invalid) == 0})

validity_summary = pd.DataFrame(validity_results)

display(validity_summary.style.hide(axis="index"))

print(
    "All coded variables contain permitted values:",
    validity_summary["Valid"].all())

Variable,Observed_Min,Observed_Max,Invalid_Values,Valid
HighBP,0,1,None,True
HighChol,0,1,None,True
CholCheck,0,1,None,True
Smoker,0,1,None,True
Stroke,0,1,None,True
HeartDiseaseorAttack,0,1,None,True
PhysActivity,0,1,None,True
Fruits,0,1,None,True
Veggies,0,1,None,True
HvyAlcoholConsump,0,1,None,True


All coded variables contain permitted values: True


## 6. Investigation of Potential Outliers

BMI is the only continuous variable in the dataset. Its values are reviewed to identify unusually low or high observations. These unusual values are investigated as possible outliers, but they are not automatically treated as errors or removed.

In [6]:
bmi_summary = df["BMI"].describe(percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]).to_frame(name="BMI")

q1 = df["BMI"].quantile(0.25)
q3 = df["BMI"].quantile(0.75)
iqr = q3 - q1

lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr

potential_bmi_outliers = (
    (df["BMI"] < lower_bound)
    | (df["BMI"] > upper_bound)).sum()

display(bmi_summary)

print(f"IQR lower boundary: {lower_bound:.2f}")
print(f"IQR upper boundary: {upper_bound:.2f}")
print(f"Potential BMI outliers: {potential_bmi_outliers:,}")
print(f"Observed BMI range: {df['BMI'].min()} to {df['BMI'].max()}")

,BMI
count,253680.000000
mean,28.382364
std,6.608694
min,12.000000
1%,18.000000
25%,24.000000
50%,27.000000
75%,31.000000
99%,50.000000
max,98.000000


IQR lower boundary: 13.50
IQR upper boundary: 41.50
Potential BMI outliers: 9,847
Observed BMI range: 12 to 98


### BMI Outlier Decision

The IQR method identified 9,847 BMI observations outside the range of 13.5 to 41.5. However, this method only identifies values that are unusual compared with most of the dataset, it does not prove that they are incorrect.

The observed BMI values range from 12 to 98 and were already retained during Phase I because there was no evidence that they resulted from data-entry errors. They will remain in the dataset because unusually high or low BMI values may contain useful information for diabetes risk prediction. Their treatment through scaling or discretisation will be considered after the shared training and test data have been established.

## 7. Duplicate-Row Review

The dataset contains rows with identical values across all 22 columns. Because the survey does not provide a unique participant identifier, identical rows cannot be confirmed as accidental duplicate records. Different respondents may have provided the same answers, particularly because most variables contain only a small number of possible values.

In [7]:
matching_rows = df.duplicated().sum()
matching_percentage = matching_rows / len(df) * 100

print(f"Exact matching rows: {matching_rows:,}")
print(f"Percentage of dataset: {matching_percentage:.2f}%")
print(f"Rows remaining if removed: {len(df) - matching_rows:,}")

Exact matching rows: 24,206
Percentage of dataset: 9.54%
Rows remaining if removed: 229,474


### Duplicate-Row Decision

The 24,206 matching rows represent approximately 9.54% of the dataset. These rows will be retained because there is no participant identifier that can confirm they are duplicate submissions. Removing them could incorrectly discard valid respondents and change the original class distribution.

## 8. Proposed Preprocessing Plan

The validation results show that the dataset does not require missing value imputation, duplicate removal, or correction of invalid category codes. The remaining preprocessing decisions therefore focus on representing the different feature types appropriately and preparing the data for fair model training.

Transformations that learn information from the dataset will be fitted using the training data only. The same fitted transformations will then be applied to the test data.

In [8]:
preprocessing_plan = pd.DataFrame({
    "Feature_Group": [
        "Binary",
        "Ordinal",
        "Continuous",
        "Count-based",
        "Target imbalance"
    ],
    "Variables": [
        ", ".join(binary_features),
        ", ".join(ordinal_features),
        ", ".join(continuous_features),
        ", ".join(count_features),
        "Diabetes_binary"
    ],
    "Proposed_Handling": [
        "Retain existing 0/1 encoding",
        "Retain ordered codes and preserve their ranking",
        "Retain BMI; assess scaling or discretisation after the data split",
        "Retain 0–30 values; assess scaling or discretisation after the data split",
        "Apply any balancing method to the training data only"
    ],
    "Reason": [
        "Values already represent two valid categories",
        "The numbers represent ordered categories rather than exact measurements",
        "BMI is valid but has a wider numerical range than most predictors",
        "These variables represent numbers of unhealthy days",
        "The test data must retain the original class distribution"
    ]
})

display(preprocessing_plan.style.hide(axis="index"))

Feature_Group,Variables,Proposed_Handling,Reason
Binary,"HighBP, HighChol, CholCheck, Smoker, Stroke, HeartDiseaseorAttack, PhysActivity, Fruits, Veggies, HvyAlcoholConsump, AnyHealthcare, NoDocbcCost, DiffWalk, Sex",Retain existing 0/1 encoding,Values already represent two valid categories
Ordinal,"GenHlth, Age, Education, Income",Retain ordered codes and preserve their ranking,The numbers represent ordered categories rather than exact measurements
Continuous,BMI,Retain BMI; assess scaling or discretisation after the data split,BMI is valid but has a wider numerical range than most predictors
Count-based,"MentHlth, PhysHlth",Retain 0–30 values; assess scaling or discretisation after the data split,These variables represent numbers of unhealthy days
Target imbalance,Diabetes_binary,Apply any balancing method to the training data only,The test data must retain the original class distribution
